# DTS103TC Coursework — Solution Notebook

**Author:** Jingyu Ye  
**Student ID:** 2470680  
**Module:** DTS103TC Data Structure and Algorithms

---

This notebook contains the Python implementations for the four coursework tasks. 
The full report (with algorithm design, correctness proofs, recurrence analysis, 
execution traces, and complexity discussions) is provided as a separate PDF.

**Contents:**
- Task 1: Maximum Subarray Sum via Divide and Conquer
- Task 2: Pending Order Tracker with Top-Product Queries
- Task 3: Package Delivery Scheduling
- Task 4: Minimum Delivery Time on a Grid

## Task 1: Maximum Subarray Sum via Divide and Conquer

**Problem.** Given an integer array `orders_change` representing per-period 
changes in pending orders, find the maximum sum over any non-empty contiguous 
subarray.

**Approach.** Divide and conquer with a constant-time combine step. 
Each recursive call returns a 4-tuple `(total_sum, max_prefix, max_suffix, max_subarray)` 
describing its segment, which is sufficient for the parent to combine results 
in O(1) time. The recurrence T(n) = 2T(n/2) + O(1) solves to **Θ(n)**, 
asymptotically optimal.

In [2]:
from typing import List, Tuple

def max_order_increase(orders_change: List[int]) -> int:
    """
    Compute the maximum total change over any non-empty contiguous
    subarray of `orders_change` using a divide-and-conquer algorithm
    with a constant-time combine step.

    Parameters
    ----------
    orders_change : List[int]
        Per-period changes in the number of pending orders. Positive
        values represent new orders; negative values represent orders
        processed from the backlog.

    Returns
    -------
    int
        The maximum sum achievable by any non-empty contiguous subarray.
        Returns 0 if `orders_change` is empty (no observed time period).
    """
    # Edge case: an empty array means there is no observed time period,
    # so no contiguous subarray exists. We return 0 by convention.
    if not orders_change:
        return 0

    def solve(low: int, high: int) -> Tuple[int, int, int, int]:
        """
        Recursively solve the sub-instance orders_change[low..high].

        Returns a 4-tuple (total_sum, max_prefix, max_suffix, max_subarray)
        describing the segment, which is sufficient for the parent call
        to combine results in O(1) time.
        """
        # Base case: a single element. Every prefix, suffix, and subarray
        # of a singleton is the element itself.
        if low == high:
            value = orders_change[low]
            return value, value, value, value

        # Divide: split the segment at the midpoint.
        mid = (low + high) // 2

        # Conquer: recursively solve both halves.
        left_total,  left_prefix,  left_suffix,  left_best  = solve(low, mid)
        right_total, right_prefix, right_suffix, right_best = solve(mid + 1, high)

        # Combine in O(1):

        # The combined total is simply the sum of the two halves.
        total_sum = left_total + right_total

        # The best prefix either lies entirely in the left half, or it
        # spans the entire left half and continues into the right half.
        max_prefix = max(left_prefix, left_total + right_prefix)

        # Symmetric reasoning for the best suffix.
        max_suffix = max(right_suffix, right_total + left_suffix)

        # A crossing subarray must consist of the best suffix of the
        # left half joined to the best prefix of the right half.
        crossing_best = left_suffix + right_prefix

        # The best subarray is the best of three candidates.
        max_subarray = max(left_best, right_best, crossing_best)

        return total_sum, max_prefix, max_suffix, max_subarray

    return solve(0, len(orders_change) - 1)[3]

In [3]:
# --- Verification ---
print(max_order_increase([4, -2, 3, -1, 5, -3]))   # Expected: 9
print(max_order_increase([-3, -1, -2]))            # Expected: -1
print(max_order_increase([1]))                     # Expected: 1
print(max_order_increase([5, 4, -1, 7, 8]))        # Expected: 23

9
-1
1
23


### Task 1-2 & 1-3

See the accompanying PDF report for:

- **Task 1-2:** Step-by-step execution trace on `[4, -2, 3, -1, 5, -3]`, 
  including the full recursion tree and detailed combine steps at each node. 
  The algorithm returns **9**, corresponding to the subarray `[4, -2, 3, -1, 5]`.
- **Task 1-3:** Derivation of the recurrence `T(n) = 2T(n/2) + Θ(1)` and its 
  solution via two independent methods (recursion-tree method and Master Theorem), 
  both yielding **T(n) = Θ(n)**.

## Task 2: Pending Order Tracker with Top-Product Queries

**Problem.** Augment the original hash-table-based `ProductTracker` with an 
efficient `top_product` operation that returns the product with the largest 
number of pending orders.

**Approach.** Hash table + max-heap with **lazy deletion**. The hash table 
remains the authoritative source of truth; the heap stores `(-pending, product_id)` 
pairs to accelerate top-product queries. Stale entries (caused by updates) 
are discarded at query time, giving:

- `add_order`, `process_order`: **O(log n)** average
- `get_pending`: **O(1)** average
- `top_product`: **O(log n)** amortised

In [4]:
import heapq


class ProductTracker:
    def __init__(self):
        # Authoritative store: product_id -> current pending orders.
        # The hash table is the single source of truth; the heap below
        # is only a hint structure used to accelerate top_product.
        self.pending = {}

        # Max-heap of (-pending, product_id); may contain stale entries
        # that are discarded lazily at query time.
        self.heap = []

    def add_order(self, product_id, quantity):
        """
        Add 'quantity' new pending orders for the given product.
        Time complexity: O(log n) average case (heap push dominates).
        """
        if quantity < 0:
            raise ValueError("quantity must be non-negative")
        if quantity == 0:
            return

        new_count = self.pending.get(product_id, 0) + quantity
        self.pending[product_id] = new_count
        # Push the updated value; any previous entry becomes stale.
        heapq.heappush(self.heap, (-new_count, product_id))

    def process_order(self, product_id, quantity):
        """
        Process 'quantity' orders for the given product.
        The pending count cannot fall below 0.
        Time complexity: O(log n) average case.
        """
        if quantity < 0:
            raise ValueError("quantity must be non-negative")

        current = self.pending.get(product_id, 0)
        new_count = max(0, current - quantity)

        if new_count == 0:
            # Remove zero-count products to keep the dictionary compact.
            # get_pending still returns 0 for absent products, preserving
            # the original semantics.
            self.pending.pop(product_id, None)
        else:
            self.pending[product_id] = new_count
            heapq.heappush(self.heap, (-new_count, product_id))

    def get_pending(self, product_id):
        """
        Return the current number of pending orders for the given product
        (0 if the product is not tracked).
        Time complexity: O(1) average case.
        """
        return self.pending.get(product_id, 0)

    def top_product(self):
        """
        Return a product_id with the largest number of pending orders.
        If no product has positive pending orders, return None.
        Lazily discards stale heap entries at the top.
        Time complexity: O(log n) amortised.
        """
        while self.heap:
            neg_count, pid = self.heap[0]  # peek without popping
            count = -neg_count
            # A heap entry is valid iff its recorded count matches the
            # authoritative dictionary value. Products absent from the
            # dictionary (pending = 0) are automatically filtered out.
            if count > 0 and self.pending.get(pid, 0) == count:
                return pid
            # Stale or zero entry --- discard and continue.
            heapq.heappop(self.heap)
        return None  # no products with positive pending orders

In [5]:
# --- Sanity check ---
tracker = ProductTracker()
tracker.add_order(101, 10)
tracker.add_order(202, 25)
tracker.add_order(303, 15)
assert tracker.top_product() == 202, "expected 202 (25 pending)"
print("After adding: top =", tracker.top_product())

tracker.process_order(202, 25)             # 202 drops to 0, removed
assert tracker.top_product() == 303, "expected 303 (15 pending)"
print("After processing 202: top =", tracker.top_product())

tracker.add_order(101, 20)                 # 101 climbs to 30
assert tracker.top_product() == 101, "expected 101 (30 pending)"
print("After boosting 101: top =", tracker.top_product())

print("get_pending(101) =", tracker.get_pending(101))   # 30
print("get_pending(202) =", tracker.get_pending(202))   # 0
print("All assertions passed.")

After adding: top = 202
After processing 202: top = 303
After boosting 101: top = 101
get_pending(101) = 30
get_pending(202) = 0
All assertions passed.


### Task 2-2 & 2-3

See the accompanying PDF report for:

- **Task 2-2:** Average-case complexity analysis of every method, with an 
  amortised argument for `top_product` (each stale-entry cleanup is charged 
  to the update that created it).
- **Task 2-3:** Design trade-offs by query frequency:
  - Rare `top_product` → plain hash table with O(n) linear scan.
  - Frequent `top_product` → hash table + max-heap (lazy deletion) **or** 
    hash table + balanced BST for worst-case bounds.

## Task 3: Package Delivery Scheduling

**Problem.** Each package takes 1 unit of time, with a deadline and a penalty. 
Find a schedule minimising the total penalty for late packages.

**Approach.** Maximise the saved penalty (selecting a feasible on-time subset) 
via 0-1 knapsack-style DP, then subtract from the total. The 1-D rolling array 
uses O(D) space where D = min(n, max deadline).

Time: **O(n log n + nD)** — O(n²) worst case  
Space: **O(D)**

In [6]:
from typing import List

def min_penalty(deadline: List[int], penalty: List[int]) -> int:
    """
    Compute the minimum total penalty for scheduling unit-time packages
    with deadlines and penalties.

    Strategy: maximise the total penalty SAVED by selecting a feasible
    on-time subset via dynamic programming, then subtract from the total.

    Args:
        deadline: list where deadline[i] is the deadline of package i.
        penalty:  list where penalty[i] is the penalty of package i.

    Returns:
        The minimum total penalty payable (int).
    """
    if len(deadline) != len(penalty):
        raise ValueError("deadline and penalty must have the same length.")

    n = len(deadline)
    if n == 0:
        return 0

    total_penalty = sum(penalty)
    max_deadline = max(deadline)
    if max_deadline <= 0:
        # No package can be delivered on time.
        return total_penalty

    # No benefit considering more than n slots, nor beyond max deadline.
    D = min(n, max_deadline)

    # Earliest-deadline-first is optimal for any feasible on-time set.
    packages = sorted(zip(deadline, penalty), key=lambda x: x[0])

    # dp[t] = max penalty saved by delivering exactly t packages on time
    #         among the packages processed so far.
    NEG_INF = float('-inf')
    dp = [NEG_INF] * (D + 1)
    dp[0] = 0  # zero on-time packages saves zero penalty

    for d_i, p_i in packages:
        # Only slots 1 <= t <= min(d_i, D) are reachable for this package.
        # Iterate t downward so each package is used at most once.
        upper = min(d_i, D)
        for t in range(upper, 0, -1):
            if dp[t - 1] != NEG_INF:
                candidate = dp[t - 1] + p_i
                if candidate > dp[t]:
                    dp[t] = candidate

    max_saved = max(v for v in dp if v != NEG_INF)
    return total_penalty - max_saved

In [7]:
# --- Verification on the example from the brief ---
deadline = [1, 1, 2, 2, 2]
penalty  = [10, 20, 5, 25, 15]
print(min_penalty(deadline, penalty))   # Expected: 30

30


### Task 3-2 — Greedy Approach (Optional Companion Code)

The problem also admits an optimal **greedy** solution via the weighted matroid 
structure of unit-time scheduling. See the PDF report for the matroid-theoretic 
justification and the exchange argument. The greedy implementation below is 
included as a companion to verify equivalence with the DP solution.

In [8]:
import heapq
from typing import List

def min_penalty_greedy(deadline: List[int], penalty: List[int]) -> int:
    """Greedy solution using a min-heap. Equivalent in correctness to the DP."""
    if len(deadline) != len(penalty):
        raise ValueError("deadline and penalty must have the same length.")

    packages = sorted(zip(deadline, penalty), key=lambda x: x[0])
    total_penalty = sum(penalty)

    selected = []  # min-heap of penalties of currently on-time packages
    for d, p in packages:
        if d <= 0:
            continue
        heapq.heappush(selected, p)
        if len(selected) > d:
            heapq.heappop(selected)  # drop the smallest penalty

    return total_penalty - sum(selected)


# Verification: greedy and DP must agree on the example
print(min_penalty_greedy([1, 1, 2, 2, 2], [10, 20, 5, 25, 15]))   # Expected: 30

30


### Task 3-3 — Variable Delivery Times

When each package may require a different delivery time `t_i`, the problem 
generalises to a 0-1 knapsack-style DP. The matroid structure of Task 3-2 
breaks down, and the problem becomes **NP-hard in the strong sense**, so only 
a pseudo-polynomial DP is available.

**Recurrence:** sort by deadline, then
`dp[t] = max(dp[t], dp[t - t_i] + p_i)` for `t_i ≤ t ≤ min(d_i, D)`, 
where `D = min(Σt_i, max d_i)`.

Time: **O(nD)** — pseudo-polynomial  
Space: **O(D)** with rolling array

In [9]:
from typing import List

def min_penalty_variable_time(deadline: List[int],
                              penalty: List[int],
                              delivery_time: List[int]) -> int:
    """
    Minimum total penalty when packages may have different delivery times.
    Generalises Task 3-1 to a 0-1 knapsack-style DP where delivery time
    plays the role of weight and saved penalty plays the role of value.
    """
    if not (len(deadline) == len(penalty) == len(delivery_time)):
        raise ValueError("inputs must all have the same length.")

    n = len(deadline)
    if n == 0:
        return 0

    total_penalty = sum(penalty)
    max_deadline = max(deadline)
    if max_deadline <= 0:
        return total_penalty

    D = min(sum(delivery_time), max_deadline)

    # EDF order ensures the feasibility characterisation used by the recurrence.
    packages = sorted(zip(deadline, penalty, delivery_time), key=lambda x: x[0])

    # dp[t] = max saved penalty among feasible selections of total time exactly t.
    dp = [0] * (D + 1)

    for d, p, c in packages:
        if c <= 0:
            continue
        # The package, if selected on time, must finish by min(d, D).
        # Iterate t downward to ensure each package is used at most once.
        latest_finish_time = min(d, D)
        for t in range(latest_finish_time, c - 1, -1):
            if dp[t - c] + p > dp[t]:
                dp[t] = dp[t - c] + p

    max_saved = max(dp)
    return total_penalty - max_saved


# Sanity check: with all delivery_time = 1, must agree with min_penalty
print(min_penalty_variable_time(
    [1, 1, 2, 2, 2], [10, 20, 5, 25, 15], [1, 1, 1, 1, 1]
))   # Expected: 30

30


## Task 4: Minimum Delivery Time on a Grid

**Problem.** Find the minimum total travel time from `(0, 0)` to `(m-1, n-1)` 
on an m × n weighted grid with four-directional movement.

**Approach.** Dijkstra's algorithm on the implicit grid graph. Because 
movement is four-directional, the state-transition graph contains cycles 
and a naive DP would fail. The starting cell's cost is included in the 
total, consistent with the brief's example.

Time: **O(mn log(mn))**  
Space: **O(mn)**

In [10]:
import heapq
from typing import List

def min_delivery_time(grid: List[List[int]]) -> int:
    """
    Compute the minimum total travel time from the top-left to the
    bottom-right cell of a 2-D grid, where the courier may move in
    four directions and grid[i][j] is the cost of traversing cell (i, j).

    Approach: Dijkstra's algorithm on the implicit grid graph.
    Time:  O(m * n * log(m * n))
    Space: O(m * n)
    """
    if not grid or not grid[0]:
        return 0

    m, n = len(grid), len(grid[0])

    # dist[i][j] = best-known minimum travel time to reach cell (i, j)
    dist = [[float('inf')] * n for _ in range(m)]
    dist[0][0] = grid[0][0]

    # Min-heap entries: (current_distance, row, col)
    heap = [(grid[0][0], 0, 0)]

    # Four-directional moves: up, down, left, right
    directions = ((-1, 0), (1, 0), (0, -1), (0, 1))

    while heap:
        d, i, j = heapq.heappop(heap)

        # Early termination: destination extracted with its final distance
        if (i, j) == (m - 1, n - 1):
            return d

        # Skip stale entries already improved upon
        if d > dist[i][j]:
            continue

        # Relax all four neighbours
        for di, dj in directions:
            ni, nj = i + di, j + dj
            if 0 <= ni < m and 0 <= nj < n:
                new_dist = d + grid[ni][nj]
                if new_dist < dist[ni][nj]:
                    dist[ni][nj] = new_dist
                    heapq.heappush(heap, (new_dist, ni, nj))

    return dist[m - 1][n - 1]

In [11]:
# --- Verification on the example from the brief ---
grid = [[7, 2, 7, 7],
        [5, 5, 4, 2],
        [9, 1, 1, 2],
        [5, 6, 7, 2]]
print(min_delivery_time(grid))   # Expected: 20

20


### Task 4-2 — Cell Count vs. Total Travel Time

See the PDF report for the discussion and a concrete 4 × 5 counterexample 
where the minimum-time route visits **10 cells** with cost 208, while a 
shorter 8-cell route costs 305. This confirms BFS would be incorrect for 
the general weighted problem.

### Task 4-3 — Optimised Algorithm for {0, 1} Weights

When `grid[i][j] ∈ {0, 1}`, Dijkstra is asymptotically suboptimal. The 
restricted weight set admits **0-1 BFS** using a double-ended queue:

- Weight-0 edges → push neighbour to the **front** (same distance bucket).
- Weight-1 edges → push neighbour to the **back** (next distance bucket).

This eliminates the log factor incurred by heap maintenance.

Time: **O(mn)** — asymptotically optimal  
Space: **O(mn)**

In [12]:
from collections import deque
from typing import List

def min_delivery_time_01(grid: List[List[int]]) -> int:
    """
    Optimised minimum travel time for grids with weights restricted to {0, 1}.
    Uses 0-1 BFS with a deque.

    Time:  O(m * n)
    Space: O(m * n)
    """
    if not grid or not grid[0]:
        return 0

    m, n = len(grid), len(grid[0])

    dist = [[float('inf')] * n for _ in range(m)]
    dist[0][0] = grid[0][0]

    dq = deque([(0, 0)])
    directions = ((-1, 0), (1, 0), (0, -1), (0, 1))

    while dq:
        i, j = dq.popleft()

        if (i, j) == (m - 1, n - 1):
            return dist[i][j]

        for di, dj in directions:
            ni, nj = i + di, j + dj
            if 0 <= ni < m and 0 <= nj < n:
                w = grid[ni][nj]                    # weight is 0 or 1
                new_dist = dist[i][j] + w
                if new_dist < dist[ni][nj]:
                    dist[ni][nj] = new_dist
                    if w == 0:
                        dq.appendleft((ni, nj))     # zero-cost: front
                    else:
                        dq.append((ni, nj))         # unit-cost: back

    return dist[m - 1][n - 1]


# --- Sanity check on a small {0, 1} grid ---
grid_01 = [[0, 1, 1, 0],
           [1, 1, 0, 1],
           [0, 0, 0, 1],
           [1, 1, 0, 0]]
print(min_delivery_time_01(grid_01))   # Optimal path cost: 1

1
